In [ ]:
# Colab-friendly NCA pre-pre-training data generator
# Based on:
# - "Training Language Models via Neural Cellular Automata" (arXiv:2603.10055)
# - the user-provided NCA generation/tokenization codebase
#
# Usage in Colab:
#   1) Paste this whole file into a cell, or save it as nca_prepretraining_data_generator_colab.py
#   2) Run the bottom example block.
#
# What this script does:
#   - samples random discrete NCA rules
#   - rolls out trajectories on a 12x12 grid
#   - filters trajectories by gzip complexity band
#   - tokenizes each frame with 2x2 patches + <grid>, </grid>
#   - saves train/val arrays and metadata
#   - visualizes trajectories, complexity histogram, token frequencies
#
# The implementation is intentionally practical for Colab:
#   - pure PyTorch + NumPy + matplotlib
#   - no JAX/Flax requirement
#   - faithful to the paper's data generation recipe

from __future__ import annotations

import gzip
import io
import json
import math
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib import colors as mcolors
from matplotlib import animation
from torch import nn

from NCA_data import build_nca_dataset, NCAConfig, NCATokenizer


# ============================================================
# 0. Colab helper
# ============================================================

def seed_everything(seed: int = 0) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ============================================================
# 8. Visualization
# ============================================================

DEFAULT_COLORS = [
    "#ff0000", "#00ff00", "#0000ff", "#ffff00", "#ff00ff",
    "#00ffff", "#ffffff", "#8f5d00", "#444444", "#ff8800",
]


def make_discrete_cmap(num_colors: int) -> mcolors.ListedColormap:
    if num_colors <= len(DEFAULT_COLORS):
        cols = DEFAULT_COLORS[:num_colors]
    else:
        cols = [plt.cm.tab20(i / max(1, num_colors - 1)) for i in range(num_colors)]
    return mcolors.ListedColormap(cols)


def show_trajectory_grid(traj: np.ndarray, title: str = "Trajectory", max_frames: int = 12) -> None:
    T = min(len(traj), max_frames)
    cols = min(6, T)
    rows = math.ceil(T / cols)
    cmap = make_discrete_cmap(int(traj.max()) + 1)

    fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.2 * rows))
    axes = np.array(axes).reshape(rows, cols)
    for i in range(rows * cols):
        ax = axes.flat[i]
        ax.axis("off")
        if i < T:
            ax.imshow(traj[i], cmap=cmap, vmin=0, vmax=cmap.N - 1, interpolation="nearest")
            ax.set_title(f"t={i}")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def plot_gzip_hist(train_gzip: np.ndarray, val_gzip: np.ndarray, threshold_low: float, threshold_high: Optional[float]) -> None:
    plt.figure(figsize=(6, 4))
    plt.hist(train_gzip, bins=20, alpha=0.7, label="train")
    plt.hist(val_gzip, bins=20, alpha=0.5, label="val")
    plt.axvline(threshold_low, color="crimson", linestyle="--", label=f"low={threshold_low:.2f}")
    if threshold_high is not None:
        plt.axvline(threshold_high, color="darkgreen", linestyle="--", label=f"high={threshold_high:.2f}")
    plt.xlabel("gzip ratio = compressed/raw")
    plt.ylabel("count")
    plt.title("Complexity distribution of kept trajectories")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_token_frequency_loglog(input_ids: np.ndarray, ignore_value: int = -100) -> None:
    flat = input_ids.reshape(-1)
    flat = flat[flat != ignore_value]
    uniq, counts = np.unique(flat, return_counts=True)
    counts = np.sort(counts)[::-1]
    ranks = np.arange(1, len(counts) + 1)

    plt.figure(figsize=(5, 4))
    plt.loglog(ranks, counts / counts.sum())
    plt.xlabel("rank")
    plt.ylabel("normalized frequency")
    plt.title("Patch-token frequency distribution")
    plt.tight_layout()
    plt.show()


def inspect_tokenization_example(cfg: NCAConfig, traj: np.ndarray) -> None:
    tokenizer = NCATokenizer(cfg)
    frame = traj[0]
    frame_tokens = tokenizer.encode_frame(frame)
    decoded = tokenizer.decode_frame(frame_tokens[1:-1])

    print("Frame shape:", frame.shape)
    print("Tokens per frame:", len(frame_tokens))
    print("Start token:", frame_tokens[0], "End token:", frame_tokens[-1])
    print("First 16 patch tokens:", frame_tokens[1:17].tolist())
    print("Round-trip decode exact:", np.array_equal(frame, decoded))

    cmap = make_discrete_cmap(cfg.num_colors)
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(frame, cmap=cmap, vmin=0, vmax=cfg.num_colors - 1, interpolation="nearest")
    axes[0].set_title("Original frame")
    axes[0].axis("off")
    axes[1].imshow(decoded, cmap=cmap, vmin=0, vmax=cfg.num_colors - 1, interpolation="nearest")
    axes[1].set_title("Decoded from 2x2 tokens")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


def make_animation_html(traj: np.ndarray, interval_ms: int = 300):
    from IPython.display import HTML
    cmap = make_discrete_cmap(int(traj.max()) + 1)
    fig, ax = plt.subplots(figsize=(3, 3))
    im = ax.imshow(traj[0], cmap=cmap, vmin=0, vmax=cmap.N - 1, interpolation="nearest")
    ax.axis("off")

    def update(i: int):
        im.set_data(traj[i])
        ax.set_title(f"t={i}")
        return [im]

    ani = animation.FuncAnimation(fig, update, frames=len(traj), interval=interval_ms, blit=True)
    plt.close(fig)
    return HTML(ani.to_jshtml())


def visualize_dataset(cfg: NCAConfig, train: Dict[str, np.ndarray], val: Dict[str, np.ndarray]) -> None:
    print("Train input_ids shape:", train["input_ids"].shape)
    print("Train labels shape:", train["labels"].shape)
    print("Train trajectories shape:", train["trajectories"].shape)
    print("Train gzip ratio mean ± std:", float(train["gzip_ratio"].mean()), float(train["gzip_ratio"].std()))
    print("Val gzip ratio mean ± std:", float(val["gzip_ratio"].mean()), float(val["gzip_ratio"].std()))
    print("Vocabulary size:", cfg.total_vocab_size)
    print("Tokens per frame:", cfg.tokens_per_frame)

    show_trajectory_grid(train["trajectories"][0], title="Example kept NCA trajectory")
    inspect_tokenization_example(cfg, train["trajectories"][0])
    plot_gzip_hist(train["gzip_ratio"], val["gzip_ratio"], cfg.gzip_threshold_low, cfg.gzip_threshold_high)
    plot_token_frequency_loglog(train["input_ids"])


# ============================================================
# 10. Colab example
# ============================================================

if __name__ == "__main__":
    # For a quick Colab demo, keep sizes modest.
    # You can later scale train_size / val_size much higher.
    cfg = NCAConfig(
        grid_size=12,
        num_colors=10,
        temperature=1e-3,
        identity_bias=0.0,
        patch_size=2,
        seq_len=1024,
        rollout_steps=32,
        time_subsample=1,
        start_step=0,
        gzip_threshold_low=0.40,
        gzip_threshold_high=0.60,
        train_size=128,
        val_size=32,
        batch_candidate_size=64,
        max_sampling_rounds=200,
        out_dir="./nca_dataset_colab_demo",
    )

    train, val = build_nca_dataset(cfg, seed=43)
    visualize_dataset(cfg, train, val)
    # Optional animation in notebooks / Colab:
    from IPython.display import display
    display(make_animation_html(train["trajectories"][0]))


ModuleNotFoundError: No module named 'NCA_datas'